In [ ]:
import os, sys
# ── CELL 0 : Output capture — run FIRST ─────────────────────────────────────
OUT_DIR = "ps4_clustering_pyspark_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream
        self._log = log_fh
        self._is_tee = True
    def write(self, data):
        for s in (self._jupyter, self._log):
            try: s.write(data); s.flush()
            except Exception: pass
    def flush(self):
        for s in (self._jupyter, self._log):
            try: s.flush()
            except Exception: pass
    def __getattr__(self, name):
        return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] tee ->", os.path.join(OUT_DIR, "console.log"))

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe = "".join(c if c.isalnum() or c in "-_ " else "_" for c in _title).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig -> {_fname}")
                    _cnt[0] += 1
                except Exception as _e:
                    print(f"[capture] save failed: {_e}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
except Exception as _e:
    print("[capture] matplotlib not ready:", _e)



# PS4 — Fault Clustering : GATE (PySpark + Spark ML)

Full PySpark port of `PS4_FaultClustering_GATE.ipynb` with **Spark ML** clustering (KMeans, PCA, GMM, BisectingKMeans) plus **t-SNE** pipelines for visualization and clustering.

**Pipelines:** P1 KMeans | P2 PCA+KMeans | P3 t-SNE+KMeans | **P4 PCA+IF+KMeans** | **P5 PCA+IF→t-SNE+KMeans** | P6 DBSCAN | P7 GMM | P8 t-SNE comparison

**Run order:** 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → **12 → 13** → 14 → 15 → 16 → 17


In [ ]:
# ── CELL 1 : Install ────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "pyspark>=3.4", "s3fs", "pyarrow", "boto3", "mlflow>=2.13", "sagemaker-mlflow",
    "kneed", "scikit-learn>=1.4", "matplotlib", "seaborn", "pandas>=2.0", "numpy", "joblib",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + pkgs)
print("Install complete.")



In [ ]:
# ── CELL 2 : Imports ────────────────────────────────────────────────────────
import os, json, time, shutil, warnings, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import mlflow
import mlflow.spark
import mlflow.sklearn
import joblib
from kneed import KneeLocator
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, KMeans as skKMeans
from sklearn.ensemble import IsolationForest
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StructType, StructField
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.clustering import KMeans, BisectingKMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
sns.set_style("whitegrid")
print("Imports OK")



In [ ]:
# ── CELL 3 : Configuration ──────────────────────────────────────────────────
S3_BUCKET   = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
REGION      = "us-east-1"
GOLD_S3     = f"s3://{S3_BUCKET}/{GOLD_PREFIX}"

TARGET_DEVICE = "GATE"
DEVICE_TYPES  = ["TVM", "GATE", "VALIDATOR"]
DEV_COL       = "mars_device_category"
DEVICE_ID_COL = "DEVICE_ID"
DEVICE_KEY    = "DEVICE_KEY"
DATE_COL      = "event_date"
TARGET_COL    = "will_fail_3d"

RANDOM_STATE = 42
K_MIN, K_MAX = 2, 10
IF_CONTAMINATION = 0.05
DBSCAN_K_NEIGHBORS = 5
DBSCAN_MIN_SAMPLES = 5

# t-SNE (driver-side on Spark-aggregated device grain — typically few-k rows)
TSNE_PERPLEXITY = 30
TSNE_N_ITER     = 1000
TSNE_SAMPLE_MAX = None   # None = all GATE devices after aggregation
TSNE_PCA_PRE    = 50     # PCA pre-reduction before t-SNE for speed

EXCLUDE_COLS = {
    DEV_COL, DEVICE_ID_COL, "DEVICE_NAME", DATE_COL, TARGET_COL,
    "mars_device_category", "device_id", "device_name",
    "split", "event_date", "FACILITY_ID", "FACILITY_NAME",
    "STATION_ID", "STATION_NAME", "OPERATOR_ID", "BUS_ID",
    DEVICE_KEY, "TRANSIT_ARRAY_ID", "ARRAY_POSITION",
    "will_fail_3d", "will_fail_7d", "max_failure_level",
    "oos_flag", "oos_event_count",
}

# Spark / S3A
LOCAL_SPARK_TEMP_DIR = os.path.join(OUT_DIR, "spark_tmp")
LOCAL_SPARK_THREADS = "*"
LOCAL_SPARK_DRIVER_MEMORY = "16g"
LOCAL_SPARK_DRIVER_MEMORY_GB = 16
LOCAL_SHUFFLE_PARTITIONS = 32
SPARK_JARS_PACKAGES = "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
AUTO_INSTALL_LOCAL_JAVA = True
OPENJDK_VERSION = "17"

CKPT_DIR = os.path.join(OUT_DIR, "checkpoint")
SPARK_CKPT_DEVICE = os.path.join(CKPT_DIR, "gate_device_grain.parquet")
SPARK_CKPT_META   = os.path.join(CKPT_DIR, "meta.joblib")
SPARK_CKPT_VERSION = "ps4-cluster-gate-pyspark-v1"
FORCE_RELOAD = False

MLFLOW_TRACKING_URI = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
MLFLOW_EXPERIMENT   = "chicago-ps4-fault-clustering-pyspark"
MLFLOW_MODEL_NAME   = "chicago-ps4-gate-fault-clustering-pyspark"
MODEL_REGISTRY_NAME = "chicago-ps4-gate-fault-clustering-pyspark"
COMMON_TAGS = {"ps": "PS4", "city": "chicago", "target": "fault_clustering",
               "model_version": "pyspark-v1", "device_type": TARGET_DEVICE, "engine": "pyspark"}

spark = None
device_sdf = None
FEATURE_COLS = []
PIPELINE_RESULTS = {}

print(f"Config ready. OUT_DIR={OUT_DIR}")
print(f"Gold: {GOLD_S3}  |  Device: {TARGET_DEVICE}")



In [ ]:
# ── CELL 4 : MLflow ───────────────────────────────────────────────────────────
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f"MLflow experiment: {MLFLOW_EXPERIMENT}")



In [ ]:
# ── CELL 5 : Spark session + S3A ────────────────────────────────────────────
def _run_conda(requirements):
    try:
        ip = get_ipython()  # noqa: F821
    except NameError:
        ip = None
    if ip is not None:
        ip.run_line_magic("conda", f"install -y -c conda-forge {' '.join(requirements)}")
    else:
        ce = shutil.which("conda")
        if not ce:
            raise RuntimeError(f"Run: conda install -y -c conda-forge openjdk={OPENJDK_VERSION}")
        subprocess.check_call([ce, "install", "-y", "-c", "conda-forge", *requirements])

def _java_from_home(java_home):
    if not java_home:
        return None
    c = os.path.join(java_home, "bin", "java")
    return c if os.path.isfile(c) and os.access(c, os.X_OK) else None

def _discover_java():
    ch = os.environ.get("JAVA_HOME")
    cj = _java_from_home(ch)
    if cj:
        return os.path.realpath(ch), os.path.realpath(cj)
    for c in [shutil.which("java"), os.path.join(sys.prefix, "bin", "java"),
              "/usr/lib/jvm/java-17-amazon-corretto/bin/java"]:
        if c and os.path.isfile(c) and os.access(c, os.X_OK):
            rj = os.path.realpath(c)
            rh = os.path.dirname(os.path.dirname(rj))
            if _java_from_home(rh):
                return rh, rj
    return None, None

def ensure_local_java():
    jh, je = _discover_java()
    if je is None and AUTO_INSTALL_LOCAL_JAVA:
        print(f"Installing OpenJDK {OPENJDK_VERSION} ...")
        _run_conda([f"openjdk={OPENJDK_VERSION}"])
        jh, je = _discover_java()
    if je is None:
        raise RuntimeError(f"Install Java: conda install openjdk={OPENJDK_VERSION}")
    os.environ["JAVA_HOME"] = jh
    jb = os.path.join(jh, "bin")
    if jb not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = jb + os.pathsep + os.environ.get("PATH", "")
    print(f"JAVA_HOME: {jh}")
    return jh

def spark_path(uri):
    return "s3a://" + uri[len("s3://"):] if uri.startswith("s3://") else uri

def _apply_s3a_hadoop_conf(_spark):
    """Override S3A duration defaults like '60s' that break Hadoop 3.3.x parsers."""
    _s3a_numeric = {
        "fs.s3a.connection.timeout": "200000",
        "fs.s3a.connection.establish.timeout": "30000",
        "fs.s3a.connection.request.timeout": "60000",
        "fs.s3a.connection.acquisition.timeout": "60000",
        "fs.s3a.connection.idle.time": "60000",
        "fs.s3a.connection.ttl": "300000",
        "fs.s3a.socket.timeout": "60000",
        "fs.s3a.threads.keepalivetime": "60",
        "fs.s3a.multipart.purge.age": "86400000",
    }
    for key, value in _s3a_numeric.items():
        try:
            _spark.conf.set(f"spark.hadoop.{key}", value)
        except Exception as exc:
            print(f"[S3A] spark.conf {key}: {type(exc).__name__}")
    try:
        _hconf = _spark.sparkContext._jsc.hadoopConfiguration()
        for key, value in _s3a_numeric.items():
            _hconf.set(key, value)
    except Exception as exc:
        print(f"[S3A] hadoopConfiguration override: {type(exc).__name__}")
    try:
        _spark._jvm.org.apache.hadoop.fs.FileSystem.closeAll()
    except Exception:
        pass

def build_spark():
    ensure_local_java()
    os.makedirs(LOCAL_SPARK_TEMP_DIR, exist_ok=True)
    esa = os.environ.get("PYSPARK_SUBMIT_ARGS", "pyspark-shell")
    if "--driver-memory" not in esa:
        if not esa.strip().endswith("pyspark-shell"):
            esa = esa.strip() + " pyspark-shell"
        os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-memory {LOCAL_SPARK_DRIVER_MEMORY} {esa}"
    sess = (
        SparkSession.builder.appName("PS4-GATE-Clustering-PySpark")
        .master(f"local[{LOCAL_SPARK_THREADS}]")
        .config("spark.driver.memory", LOCAL_SPARK_DRIVER_MEMORY)
        .config("spark.driver.maxResultSize", f"{max(1, LOCAL_SPARK_DRIVER_MEMORY_GB // 2)}g")
        .config("spark.local.dir", LOCAL_SPARK_TEMP_DIR)
        .config("spark.jars.packages", SPARK_JARS_PACKAGES)
        .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.timeout", "200000")
        .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000")
        .config("spark.hadoop.fs.s3a.connection.request.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.acquisition.timeout", "60000")
        .config("spark.hadoop.fs.s3a.connection.idle.time", "60000")
        .config("spark.hadoop.fs.s3a.connection.ttl", "300000")
        .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
        .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60")
        .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400000")
        .getOrCreate()
    )
    for k, v in {
        "spark.sql.adaptive.enabled": "true",
        "spark.sql.parquet.filterPushdown": "true",
        "spark.sql.shuffle.partitions": str(LOCAL_SHUFFLE_PARTITIONS),
        "spark.sql.execution.arrow.pyspark.enabled": "true",
    }.items():
        try:
            sess.conf.set(k, v)
        except Exception:
            pass
    _apply_s3a_hadoop_conf(sess)
    sess.sparkContext.setLogLevel("WARN")
    return sess

def ensure_spark_alive(recreate=False):
    global spark
    _s = globals().get("spark")
    if _s is not None and not recreate:
        try:
            _s.range(1).count()
            _apply_s3a_hadoop_conf(_s)
            return _s
        except Exception as e:
            print(f"[WARN] Spark dead ({type(e).__name__}) — recreating")
    if recreate or _s is not None:
        try:
            a = SparkSession.getActiveSession()
            if a:
                a.stop()
        except Exception:
            pass
    spark = build_spark()
    return spark

def read_parquet_s3(path):
    ensure_spark_alive()
    _apply_s3a_hadoop_conf(spark)
    p = spark_path(path)
    print(f"[S3A] Reading {p}")
    return spark.read.parquet(p)

spark = ensure_spark_alive()
print(f"Spark {spark.version}  master={spark.sparkContext.master}")



In [ ]:
# ── CELL 6 : Load Gold + filter GATE (PySpark) ──────────────────────────────
ensure_spark_alive()
_meta = joblib.load(SPARK_CKPT_META) if (not FORCE_RELOAD and os.path.exists(SPARK_CKPT_META)) else None
if _meta and _meta.get("version") == SPARK_CKPT_VERSION and os.path.exists(SPARK_CKPT_DEVICE):
    print(f"[CKPT] Loading device-grain checkpoint {SPARK_CKPT_DEVICE}")
    device_sdf = spark.read.parquet(SPARK_CKPT_DEVICE)
    FEATURE_COLS = _meta["FEATURE_COLS"]
    print(f"  Devices: {device_sdf.count():,}  |  Features: {len(FEATURE_COLS)}")
else:
    df_full = read_parquet_s3(GOLD_S3)
    print(f"Loaded: {df_full.count():,} rows x {len(df_full.columns)} cols")

    if DATE_COL in df_full.columns:
        df_full = df_full.withColumn(DATE_COL, F.to_timestamp(F.col(DATE_COL)))
    if DEV_COL in df_full.columns:
        df_full = df_full.withColumn(DEV_COL, F.upper(F.trim(F.col(DEV_COL).cast("string"))))
        print("Device counts:")
        df_full.groupBy(DEV_COL).count().orderBy(F.desc("count")).show(10, truncate=False)

    gate_sdf = df_full.filter(F.col(DEV_COL) == TARGET_DEVICE)
    print(f"{TARGET_DEVICE} rows: {gate_sdf.count():,}")

    _excl = {c.lower() for c in EXCLUDE_COLS}
    FEATURE_COLS = [
        c for c in gate_sdf.columns
        if c.lower() not in _excl
        and dict(gate_sdf.dtypes)[c] in ("int", "bigint", "double", "float", "decimal", "long", "short")
    ]
    print(f"Feature columns: {len(FEATURE_COLS)}")

    _agg_exprs = [F.mean(F.col(c)).alias(c) for c in FEATURE_COLS]
    if TARGET_COL in gate_sdf.columns:
        _agg_exprs.append(F.mean(F.col(TARGET_COL)).alias(TARGET_COL))

    if DEVICE_KEY not in gate_sdf.columns:
        raise ValueError(f"{DEVICE_KEY} not found — required for device-grain aggregation")

    device_sdf = gate_sdf.groupBy(DEVICE_KEY).agg(*_agg_exprs)

    if DEVICE_ID_COL in gate_sdf.columns:
        _id_map = gate_sdf.select(DEVICE_KEY, DEVICE_ID_COL).dropDuplicates([DEVICE_KEY])
        device_sdf = device_sdf.join(_id_map, on=DEVICE_KEY, how="left")

    for c in FEATURE_COLS:
        device_sdf = device_sdf.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))

    os.makedirs(CKPT_DIR, exist_ok=True)
    device_sdf.write.mode("overwrite").parquet(SPARK_CKPT_DEVICE)
    joblib.dump({"version": SPARK_CKPT_VERSION, "FEATURE_COLS": FEATURE_COLS}, SPARK_CKPT_META)
    print(f"Device grain: {device_sdf.count():,} unique devices — checkpoint saved")



In [ ]:
# ── CELL 7 : Spark ML preprocess pipeline ─────────────────────────────────────
ensure_spark_alive()

IMP_COLS = [f"{c}__imp" for c in FEATURE_COLS]
FEAT_COL = "features_scaled"
TSNE_FEAT_COL = "tsne_features"

def build_preprocess_stages():
    imputer = Imputer(inputCols=FEATURE_COLS, outputCols=IMP_COLS, strategy="median")
    assembler = VectorAssembler(inputCols=IMP_COLS, outputCol="features_raw", handleInvalid="keep")
    scaler = StandardScaler(inputCol="features_raw", outputCol=FEAT_COL, withStd=True, withMean=True)
    return [imputer, assembler, scaler]

def fit_preprocess(sdf):
    pipe = Pipeline(stages=build_preprocess_stages())
    model = pipe.fit(sdf)
    return model, model.transform(sdf)

def spark_full_pipeline(*tail_stages):
    """Fitted preprocess + downstream stages (MLflow requires PipelineModel)."""
    return PipelineModel(stages=preprocess_model.stages + list(tail_stages))

def spark_silhouette(sdf, features_col=FEAT_COL, prediction_col="cluster"):
    ev = ClusteringEvaluator(featuresCol=features_col, predictionCol=prediction_col,
                             metricName="silhouette", distanceMeasure="squaredEuclidean")
    return ev.evaluate(sdf)

def spark_elbow(sdf, features_col=FEAT_COL, label=""):
    kr = list(range(K_MIN, K_MAX + 1))
    wcss = []
    for k in kr:
        km = KMeans(k=k, seed=RANDOM_STATE, featuresCol=features_col, maxIter=50)
        m = km.fit(sdf)
        wcss.append(float(m.summary.trainingCost))
    kl = KneeLocator(kr, wcss, curve="convex", direction="decreasing")
    opt = kl.knee if kl.knee else K_MIN + 1
    print(f"  {label} optimal K = {opt}")
    return int(opt), kr, wcss

def pca95_fit_transform(scaled_sdf, input_col=FEAT_COL):
    k_max = min(len(FEATURE_COLS), 50)
    pca_full = PCA(k=k_max, inputCol=input_col, outputCol="pca_full").fit(scaled_sdf)
    ev = pca_full.explainedVariance.toArray()
    cum = np.cumsum(ev)
    n = int(np.argmax(cum >= 0.95) + 1)
    pca = PCA(k=n, inputCol=input_col, outputCol="pca_features")
    pca_model = Pipeline(stages=[pca]).fit(scaled_sdf)
    out = pca_model.transform(scaled_sdf)
    return out, n, cum, pca_model

def sdf_to_pdf(sdf, max_rows=None):
    n = sdf.count()
    if max_rows and n > max_rows:
        sdf = sdf.sample(withReplacement=False, fraction=min(1.0, max_rows / max(n, 1)), seed=RANDOM_STATE)
    return sdf.toPandas()

def add_tsne_columns(pdf, feature_matrix, prefix="tsne"):
    n_samples = feature_matrix.shape[0]
    perplexity = min(TSNE_PERPLEXITY, max(5, (n_samples - 1) // 3))
    print(f"  t-SNE: n={n_samples:,}  perplexity={perplexity}")
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=TSNE_N_ITER,
                random_state=RANDOM_STATE, init="pca", learning_rate="auto")
    xy = tsne.fit_transform(feature_matrix)
    pdf[f"{prefix}_1"] = xy[:, 0]
    pdf[f"{prefix}_2"] = xy[:, 1]
    return pdf, xy

TSNE_CACHE = {}
IF_CACHE = {}

def get_or_compute_tsne(feature_matrix, meta, cache_key, prefix="tsne", pre_pca=True):
    """Reuse cached t-SNE embedding when available; otherwise compute and cache."""
    if cache_key in TSNE_CACHE:
        cached_pdf, cached_xy = TSNE_CACHE[cache_key]
        print(f"  [t-SNE] Reusing cache '{cache_key}' ({len(cached_pdf):,} devices)")
        return cached_pdf.copy(), cached_xy
    X_in = feature_matrix
    if pre_pca and X_in.shape[1] > TSNE_PCA_PRE:
        from sklearn.decomposition import PCA as skPCA
        X_in = skPCA(n_components=TSNE_PCA_PRE, random_state=RANDOM_STATE).fit_transform(X_in)
        print(f"  Pre-PCA for t-SNE: {feature_matrix.shape[1]} -> {TSNE_PCA_PRE} dims")
    pdf = pd.DataFrame({DEVICE_KEY: meta[DEVICE_KEY]})
    if TARGET_COL in meta:
        pdf[TARGET_COL] = meta[TARGET_COL]
    if DEVICE_ID_COL in meta:
        pdf[DEVICE_ID_COL] = meta[DEVICE_ID_COL]
    pdf, xy = add_tsne_columns(pdf, X_in, prefix=prefix)
    TSNE_CACHE[cache_key] = (pdf.copy(), xy)
    return pdf, xy

def collect_pca_matrix(pca_sdf, col="pca_features"):
    rows = pca_sdf.select(
        DEVICE_KEY, vector_to_array(F.col(col)).alias("_pa"),
        *([TARGET_COL] if TARGET_COL in pca_sdf.columns else []),
        *([DEVICE_ID_COL] if DEVICE_ID_COL in pca_sdf.columns else []),
    ).collect()
    Xp = np.array([r["_pa"] for r in rows], dtype=float)
    meta = {DEVICE_KEY: [r[DEVICE_KEY] for r in rows]}
    if TARGET_COL in pca_sdf.columns:
        meta[TARGET_COL] = [float(r[TARGET_COL]) for r in rows]
    if DEVICE_ID_COL in pca_sdf.columns:
        meta[DEVICE_ID_COL] = [r[DEVICE_ID_COL] for r in rows]
    return Xp, meta

def run_isolation_forest(X, contamination=IF_CONTAMINATION):
    ifm = IsolationForest(contamination=contamination, random_state=RANDOM_STATE, n_jobs=-1)
    lbl = ifm.fit_predict(X)
    score = ifm.score_samples(X)
    return lbl, score, ifm

def pdf_tsne_to_spark(pdf, id_col=DEVICE_KEY):
    cols = [id_col, "tsne_1", "tsne_2"]
    if TARGET_COL in pdf.columns:
        cols.append(TARGET_COL)
    if DEVICE_ID_COL in pdf.columns:
        cols.append(DEVICE_ID_COL)
    tsdf = spark.createDataFrame(pdf[cols])
    assembler = VectorAssembler(inputCols=["tsne_1", "tsne_2"], outputCol=TSNE_FEAT_COL)
    return assembler.transform(tsdf)

def collect_feature_matrix(scaled_sdf, features_col=FEAT_COL):
    rows = scaled_sdf.select(
        DEVICE_KEY, vector_to_array(F.col(features_col)).alias("_arr"),
        *([TARGET_COL] if TARGET_COL in scaled_sdf.columns else []),
        *([DEVICE_ID_COL] if DEVICE_ID_COL in scaled_sdf.columns else []),
    ).collect()
    keys = [r[DEVICE_KEY] for r in rows]
    X = np.array([r["_arr"] for r in rows], dtype=float)
    meta = {DEVICE_KEY: keys}
    if TARGET_COL in scaled_sdf.columns:
        meta[TARGET_COL] = [float(r[TARGET_COL]) for r in rows]
    if DEVICE_ID_COL in scaled_sdf.columns:
        meta[DEVICE_ID_COL] = [r[DEVICE_ID_COL] for r in rows]
    return X, meta

preprocess_model, scaled_sdf = fit_preprocess(device_sdf)
_n_dev = scaled_sdf.count()
print(f"Preprocessed {TARGET_DEVICE}: {_n_dev:,} devices x {len(FEATURE_COLS)} features")



In [ ]:
# ── CELL 8 : Visualization helpers ──────────────────────────────────────────

def _cluster_summary(df_sub, cl_col, feat):
    rows = []
    total = max(len(df_sub), 1)
    _avail = [c for c in feat if c in df_sub.columns]
    _means = df_sub.groupby(cl_col)[_avail].mean() if _avail else None
    for cl in sorted(df_sub[cl_col].unique()):
        mask = df_sub[cl_col] == cl
        size = int(mask.sum())
        fail = float(df_sub.loc[mask, TARGET_COL].mean()) if TARGET_COL in df_sub.columns and size else float("nan")
        dcnt = int(df_sub.loc[mask, DEVICE_ID_COL].nunique()) if DEVICE_ID_COL in df_sub.columns else -1
        if _means is not None and cl in _means.index and len(_avail) >= 3:
            _gm = df_sub[_avail].mean()
            top3 = ", ".join((_means.loc[cl] - _gm).abs().nlargest(3).index.tolist())
        else:
            top3 = ""
        label = "Noise" if cl == -1 else str(cl)
        rows.append({"Cluster": label, "Size": size, "Pct%": round(100 * size / total, 1),
                     "FailRate": f"{fail:.4f}" if fail == fail else "N/A", "Devices": dcnt, "TopFeatures": top3})
    return pd.DataFrame(rows)

def _merge_features(pdf_cluster):
    dev = device_sdf.toPandas()
    cols = [DEVICE_KEY] + [c for c in FEATURE_COLS if c in dev.columns]
    return pdf_cluster.merge(dev[cols], on=DEVICE_KEY, how="left")

def _print_cluster_summary(sum_df, pipe_name):
    print(f"\n{'─'*60}\n  CLUSTER SUMMARY STATISTICS — {pipe_name}\n{'─'*60}")
    print(sum_df.to_string(index=False))

def _profile(df_sub, cl_col, feat, lbl, top_n=15):
    print(f"\n-- {lbl} profile (mean, top {top_n}) --")
    _avail = [c for c in feat if c in df_sub.columns]
    if not _avail:
        print("  (no features available)")
        return
    _gm = df_sub[_avail].mean()
    _cm = df_sub.groupby(cl_col)[_avail].mean()
    _disc = (_cm - _gm).abs().max(axis=0).sort_values(ascending=False).head(top_n).index.tolist()
    print(df_sub.groupby(cl_col)[_disc].mean().round(3).T.to_string())
    if DEVICE_ID_COL in df_sub.columns:
        for cl in sorted(df_sub[cl_col].unique()):
            m = df_sub[cl_col] == cl
            tag = "Noise" if cl == -1 else str(cl)
            print(f"  Cluster {tag}: {m.sum():,} rows | {df_sub.loc[m, DEVICE_ID_COL].nunique()} devices")

def _driver_pca_variance(sdf, features_col=FEAT_COL):
    from sklearn.decomposition import PCA as skPCA
    X, _ = collect_feature_matrix(sdf)
    if X.size == 0:
        return np.array([1.0]), 1
    _k = min(X.shape[0], X.shape[1])
    full = skPCA(n_components=_k, random_state=RANDOM_STATE).fit(X)
    cum = np.cumsum(full.explained_variance_ratio_)
    n95 = int(np.argmax(cum >= 0.95)) + 1 if len(cum) else 1
    return cum, n95

def _spark_pca2_scatter(sdf, features_col, pred_sdf):
    from pyspark.ml.feature import PCA as _PCA2
    _m = _PCA2(k=2, inputCol=features_col, outputCol="pca2").fit(sdf)
    _viz = _m.transform(sdf.select(DEVICE_KEY, features_col)).join(
        pred_sdf.select(DEVICE_KEY, "cluster"), DEVICE_KEY)
    _arr = _viz.select(vector_to_array("pca2").alias("a"), "cluster").toPandas()
    return _arr["a"].apply(lambda x: x[0]), _arr["a"].apply(lambda x: x[1])

def _print_device_list(df_sub, cl_col, pipe_name):
    if DEVICE_ID_COL not in df_sub.columns:
        print("  (no DEVICE_ID_COL — skip device list)")
        return
    print(f"\n{'='*60}\n  DEVICE IDs — {pipe_name}\n{'='*60}")
    for cl in sorted(df_sub[cl_col].unique()):
        ids = sorted(df_sub.loc[df_sub[cl_col] == cl, DEVICE_ID_COL].dropna().astype(str).unique())
        preview = ", ".join(ids[:15])
        suffix = f" ... +{len(ids)-15}" if len(ids) > 15 else ""
        print(f"  Cluster {cl}: {len(ids)} devices | {preview}{suffix}")

def _save(fig, fname):
    p = os.path.join(OUT_DIR, fname)
    fig.savefig(p, dpi=100, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  Saved: {p}")
    return p

def _ax_var(ax, cum, n):
    ax.plot(range(1, len(cum) + 1), cum * 100, "b-", lw=1.5)
    ax.axhline(95, color="red", ls="--", lw=1, label="95%")
    ax.axvline(n, color="orange", ls="--", lw=1, label=f"n={n}")
    ax.set(xlabel="Components", ylabel="Cumul. Variance %",
            title="Variance Explained by PCA Components")
    ax.legend(fontsize=8)

def _ax_elbow(ax, kr, wcss, opt, title):
    ax.plot(kr, wcss, "bo-", ms=5)
    ax.axvline(opt, color="red", ls="--", lw=1.5, label=f"K={opt}")
    ax.set(xlabel="K", ylabel="WCSS/Cost", title=title)
    ax.legend()

def _ax_cl(ax, x, y, cl, nc, title, xl="Dim1", yl="Dim2"):
    pal = sns.color_palette("tab10", max(nc, 1))
    cl = np.asarray(cl)
    for c in sorted(np.unique(cl)):
        m = cl == c
        lbl = f"Noise({m.sum()})" if c == -1 else f"C{c}({m.sum()})"
        ax.scatter(x[m], y[m], c=[pal[int(c) % 10] if c != -1 else "#999999"], alpha=0.45, s=8, label=lbl)
    ax.set(title=title, xlabel=xl, ylabel=yl)
    ax.legend(markerscale=2, fontsize=7, ncol=2)

def _ax_bar(ax, cl, title="Number of Data Points in Each Cluster"):
    vals = pd.Series(cl).value_counts().sort_index()
    pal = sns.color_palette("tab10", len(vals))
    ax.bar([str(v) for v in vals.index], vals.values, color=[pal[i % 10] for i in range(len(vals))])
    for i, v in enumerate(vals.values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
    ax.set(title=title, xlabel="Cluster", ylabel="Count")

def _ax_devcount(ax, df_sub, cl_col, title="Unique Devices per Cluster"):
    if DEVICE_ID_COL not in df_sub.columns:
        ax.text(0.5, 0.5, "No device col", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        return
    dcnt = df_sub.groupby(cl_col)[DEVICE_ID_COL].nunique()
    pal = sns.color_palette("tab10", len(dcnt))
    clabels = [f"C{c}" if c != -1 else "Noise" for c in dcnt.index]
    ax.bar(clabels, dcnt.values, color=[pal[i % 10] for i in range(len(dcnt))])
    for i, v in enumerate(dcnt.values):
        ax.text(i, v, str(v), ha="center", va="bottom", fontsize=8)
    ax.set(title=title, xlabel="Cluster", ylabel="Unique Devices")

def _ax_timeline(ax, df_sub, cl_col, title="Monthly Records per Cluster"):
    if DATE_COL and DATE_COL in df_sub.columns:
        try:
            _tmp = df_sub[[cl_col, DATE_COL]].copy()
            _tmp["_ym"] = pd.to_datetime(_tmp[DATE_COL], errors="coerce").dt.to_period("M").astype(str)
            _ct = _tmp.groupby(["_ym", cl_col]).size().unstack(fill_value=0)
            pal = sns.color_palette("tab10", _ct.shape[1])
            for i, c in enumerate(_ct.columns):
                lbl = f"C{c}" if c != -1 else "Noise"
                ax.plot(_ct.index, _ct[c], label=lbl, color=pal[i % 10], lw=1.5)
            ax.set(title=title, xlabel="Month", ylabel="Records")
            ax.tick_params(axis="x", rotation=45, labelsize=6)
            ax.legend(fontsize=7, ncol=2)
        except Exception as _e:
            ax.text(0.5, 0.5, f"Timeline err:\n{_e}", ha="center", va="center",
                    transform=ax.transAxes, fontsize=7)
            ax.set_title(title)
    else:
        ax.text(0.5, 0.5, "No date col (device grain)", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)

def _ax_anom(ax, df_sub, cl_col, title):
    if TARGET_COL in df_sub.columns:
        ar = df_sub.groupby(cl_col)[TARGET_COL].mean()
        ax.bar(ar.index.astype(str), ar.values, color=sns.color_palette("tab10", len(ar)))
        ax.set(title=title, xlabel="Cluster", ylabel="Failure Rate")
    else:
        ax.text(0.5, 0.5, "No target", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)

def _ax_heatmap(ax, df_sub, cl_col, feat, title, top_n=12):
    _avail = [c for c in feat if c in df_sub.columns][:top_n]
    if not _avail:
        ax.set_title(title)
        return
    _means = df_sub.groupby(cl_col)[_avail].mean()
    _rng = (_means.max() - _means.min()).replace(0, 1)
    _norm = (_means - _means.min()) / _rng
    sns.heatmap(_norm.T, ax=ax, cmap="YlOrRd", linewidths=0.3, cbar_kws={"shrink": 0.6})
    ax.set(title=title)
    ax.tick_params(axis="y", labelsize=7)

def _ax_bic(ax, kr, bics, opt):
    ax.plot(kr, bics, "bo-", ms=5)
    ax.axvline(opt, color="red", ls="--", label=f"K={opt}")
    ax.set(title="BIC vs K", xlabel="K", ylabel="BIC")
    ax.legend(fontsize=8)

def _ax_kdist(ax, kdist, knee, eps, k_neighbors=DBSCAN_K_NEIGHBORS):
    ax.plot(kdist, lw=1.2, color="#1565C0")
    if knee is not None:
        ax.axvline(knee, color="red", ls="--", label=f"knee eps={eps:.3f}")
    ax.set(title="k-Distance Plot", xlabel="Points (sorted)", ylabel=f"{k_neighbors}-NN dist")
    ax.legend(fontsize=8)

def _ax_if_hist(ax, if_score, if_lbl, title="IF Anomaly Score Distribution"):
    ax.hist(if_score[if_lbl == 1], bins=40, color="#1565C0", alpha=0.6, label="Inlier")
    ax.hist(if_score[if_lbl == -1], bins=40, color="#E53935", alpha=0.6, label="Outlier")
    ax.set(title=title, xlabel="IF score")
    ax.legend(fontsize=8)

def _ax_summary_text(ax, df_sum, title="Cluster Summary Statistics"):
    ax.axis("off")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.text(0.02, 0.97, df_sum.to_string(index=False), transform=ax.transAxes, va="top", ha="left",
            fontsize=7.5, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f4f8", alpha=0.9))

def _render_dashboard_3x3(pdf_cluster, sum_df, pipe_name, suptitle, fname, panels):
    """Render 3x3 dashboard. panels: list of 9 callables(ax) or None."""
    pdf_full = _merge_features(pdf_cluster)
    _print_cluster_summary(sum_df, pipe_name)
    _print_device_list(pdf_cluster, "cluster", pipe_name)
    fig = plt.figure(figsize=(22, 15))
    fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)
    for i, fn in enumerate(panels):
        ax = fig.add_subplot(gs[i // 3, i % 3])
        if fn is not None:
            fn(ax)
        else:
            ax.axis("off")
    fig.tight_layout()
    art = _save(fig, fname)
    _profile(pdf_full, "cluster", FEATURE_COLS, pipe_name)
    return art, pdf_full

def _std_panels(pdf_cluster, pdf_full, sum_df, scatter_x, scatter_y, n_clusters,
                scatter_title, scatter_xl, scatter_yl, pca_cum, pca_n,
                elbow_kr=None, elbow_wcss=None, elbow_opt=None,
                row0=None, row1=None):
    """Standard 9-panel layout with optional custom row 0/1 panels."""
    cl = pdf_cluster["cluster"].values
    r0 = row0 if row0 else [
        (lambda ax: _ax_var(ax, pca_cum, pca_n)) if pca_cum is not None else
        (lambda ax: _ax_elbow(ax, elbow_kr, elbow_wcss, elbow_opt, f"Elbow K={elbow_opt}") if elbow_kr else None),
        lambda ax: _ax_elbow(ax, elbow_kr, elbow_wcss, elbow_opt, f"Elbow K={elbow_opt}") if (elbow_kr and pca_cum is not None) else
                   _ax_cl(ax, scatter_x, scatter_y, cl, n_clusters, scatter_title, scatter_xl, scatter_yl),
        lambda ax: _ax_cl(ax, scatter_x, scatter_y, cl, n_clusters, scatter_title, scatter_xl, scatter_yl)
              if (elbow_kr and pca_cum is not None and scatter_x is not None) else
              lambda ax: _ax_bar(ax, cl),
    ]
    r1 = row1 if row1 else [
        lambda ax: _ax_bar(ax, cl),
        lambda ax: _ax_anom(ax, pdf_full, "cluster", "Failure Rate per Cluster"),
        lambda ax: _ax_heatmap(ax, pdf_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    ]
    r2 = [
        lambda ax: _ax_devcount(ax, pdf_full, "cluster"),
        lambda ax: _ax_timeline(ax, pdf_full, "cluster"),
        lambda ax: _ax_summary_text(ax, sum_df),
    ]
    if row0 and row1:
        return row0 + row1 + r2
    if pca_cum is not None and elbow_kr is not None:
        return [
            lambda ax: _ax_var(ax, pca_cum, pca_n),
            lambda ax: _ax_elbow(ax, elbow_kr, elbow_wcss, elbow_opt, f"Elbow K={elbow_opt}"),
            lambda ax: _ax_cl(ax, scatter_x, scatter_y, cl, n_clusters, scatter_title, scatter_xl, scatter_yl),
            lambda ax: _ax_bar(ax, cl),
            lambda ax: _ax_anom(ax, pdf_full, "cluster", "Failure Rate per Cluster"),
            lambda ax: _ax_heatmap(ax, pdf_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
        ] + r2
    return [
        lambda ax: _ax_elbow(ax, elbow_kr, elbow_wcss, elbow_opt, f"Elbow K={elbow_opt}") if elbow_kr else
                   _ax_var(ax, pca_cum, pca_n),
        lambda ax: _ax_cl(ax, scatter_x, scatter_y, cl, n_clusters, scatter_title, scatter_xl, scatter_yl),
        lambda ax: _ax_var(ax, pca_cum, pca_n) if (pca_cum is not None and elbow_kr) else _ax_bar(ax, cl),
        lambda ax: _ax_bar(ax, cl),
        lambda ax: _ax_anom(ax, pdf_full, "cluster", "Failure Rate per Cluster"),
        lambda ax: _ax_heatmap(ax, pdf_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    ] + r2

def run_spark_kmeans_pipeline(sdf, features_col, pipe_name, fname, extra_params=None):
    _t0 = time.time()
    print(f"\n{'='*70}\n  {pipe_name}\n{'='*70}")
    opt, kr, wcss = spark_elbow(sdf, features_col, pipe_name)
    km = KMeans(k=opt, seed=RANDOM_STATE, featuresCol=features_col, maxIter=50)
    km_model = km.fit(sdf)
    pred_sdf = km_model.transform(sdf).withColumnRenamed("prediction", "cluster")
    sil = spark_silhouette(pred_sdf, features_col, "cluster")
    pdf = sdf_to_pdf(pred_sdf.select(DEVICE_KEY, "cluster", *([TARGET_COL] if TARGET_COL in pred_sdf.columns else []),
                                     *([DEVICE_ID_COL] if DEVICE_ID_COL in pred_sdf.columns else [])))
    _sum = _cluster_summary(pdf, "cluster", FEATURE_COLS)
    _pca_cum, _pca_n = _driver_pca_variance(sdf, features_col)
    _sx, _sy = _spark_pca2_scatter(sdf, features_col, pred_sdf)
    _panels = _std_panels(
        pdf, _merge_features(pdf), _sum, _sx, _sy, opt,
        f"PCA-2D clusters  sil={sil:.3f}", "PC1", "PC2",
        _pca_cum, _pca_n, kr, wcss, opt,
    )
    art, _ = _render_dashboard_3x3(
        pdf, _sum, pipe_name,
        f"{TARGET_DEVICE} — {pipe_name}  K={opt}  sil={sil:.3f}", fname, _panels)
    with mlflow.start_run(run_name=pipe_name.replace(" ", "_")) as run:
        mlflow.set_tags({**COMMON_TAGS, "pipeline": pipe_name})
        mlflow.log_params({"algorithm": pipe_name, "n_clusters": opt, **(extra_params or {})})
        mlflow.log_metrics({"silhouette_score": round(sil, 4), "silhouette": round(sil, 4), "n_devices": len(pdf), "optimal_k": opt})
        mlflow.log_artifact(art)
        _full = spark_full_pipeline(km_model)
        mlflow.spark.log_model(_full, "model")
        PIPELINE_RESULTS[pipe_name] = {"pdf": pdf, "sil": sil, "opt_k": opt, "pred_sdf": pred_sdf,
                                       "run_id": run.info.run_id, "flavor": "spark", "model": _full}
    print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")
    return pred_sdf, pdf, sil, opt

print("Helpers ready.")



In [ ]:
# ── CELL 9 : [GATE] P1 — Spark K-Means (plain scaled features) ───────────────
ensure_spark_alive()
_p1_sdf, _p1_pdf, _p1_sil, _p1_k = run_spark_kmeans_pipeline(
    scaled_sdf, FEAT_COL,
    f"{TARGET_DEVICE}_P1_Spark_KMeans",
    f"{TARGET_DEVICE.lower()}_p1_spark_kmeans.png",
    {"features": len(FEATURE_COLS)},
)



In [ ]:
# ── CELL 10 : [GATE] P2 — Spark PCA (95%) + K-Means ───────────────────────────
ensure_spark_alive()
_pca_sdf, _npc, _cum, _pca_model = pca95_fit_transform(scaled_sdf)
print(f"  PCA components (95% var): {_npc}")

opt, kr, wcss = spark_elbow(_pca_sdf, "pca_features", "PCA+KMeans")
km = KMeans(k=opt, seed=RANDOM_STATE, featuresCol="pca_features", maxIter=50)
_km_model = km.fit(_pca_sdf)
_p2_pred = _km_model.transform(_pca_sdf).withColumnRenamed("prediction", "cluster")
_p2_sil = spark_silhouette(_p2_pred, "pca_features", "cluster")
_p2_pdf = sdf_to_pdf(_p2_pred.select(DEVICE_KEY, "cluster", *([TARGET_COL, DEVICE_ID_COL] if TARGET_COL in _p2_pred.columns else [DEVICE_ID_COL])))
_sum = _cluster_summary(_p2_pdf, "cluster", FEATURE_COLS)
_arr = _p2_pred.select(vector_to_array("pca_features").alias("a"), "cluster").toPandas()
_sx = _arr["a"].apply(lambda x: x[0])
_sy = _arr["a"].apply(lambda x: x[1])
_panels = _std_panels(
    _p2_pdf, _merge_features(_p2_pdf), _sum, _sx, _sy, opt,
    f"PC1-PC2  sil={_p2_sil:.3f}", "PC1", "PC2", _cum, _npc, kr, wcss, opt)
_art = _render_dashboard_3x3(
    _p2_pdf, _sum, f"{TARGET_DEVICE} P2 PCA+KMeans",
    f"{TARGET_DEVICE} — P2 PCA+KMeans  n_pca={_npc}  K={opt}  sil={_p2_sil:.3f}",
    f"{TARGET_DEVICE.lower()}_p2_pca_kmeans.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P2_PCA_KMeans") as run:
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "PCA_KMeans"})
    mlflow.log_params({"n_clusters": opt, "pca_components": _npc})
    mlflow.log_metrics({"silhouette_score": round(_p2_sil, 4), "silhouette": round(_p2_sil, 4), "n_devices": len(_p2_pdf)})
    mlflow.log_artifact(_art)
    _p2_pipe = spark_full_pipeline(_pca_model, _km_model)
    mlflow.spark.log_model(_p2_pipe, "model")
    PIPELINE_RESULTS["P2_PCA_KMeans"] = {"pdf": _p2_pdf, "sil": _p2_sil, "run_id": run.info.run_id, "flavor": "spark",
        "x": _arr["a"].apply(lambda x: x[0]), "y": _arr["a"].apply(lambda x: x[1])}



In [ ]:
# ── CELL 11 : [GATE] P3 — t-SNE + Spark K-Means ─────────────────────────────────
# Spark preprocess -> sklearn t-SNE (driver) -> Spark KMeans on t-SNE coords
ensure_spark_alive()
_t0 = time.time()
print(f"\n{'='*70}\n  {TARGET_DEVICE} P3: t-SNE + Spark K-Means\n{'='*70}")

X, meta = collect_feature_matrix(scaled_sdf)
pdf_tsne, xy = get_or_compute_tsne(X, meta, cache_key="scaled_features", prefix="tsne")

tsne_sdf = pdf_tsne_to_spark(pdf_tsne)
opt, kr, wcss = spark_elbow(tsne_sdf, TSNE_FEAT_COL, "t-SNE+KMeans")
km = KMeans(k=opt, seed=RANDOM_STATE, featuresCol=TSNE_FEAT_COL, maxIter=50)
_km_model = km.fit(tsne_sdf)
_p3_pred = _km_model.transform(tsne_sdf).withColumnRenamed("prediction", "cluster")
_p3_sil = spark_silhouette(_p3_pred, TSNE_FEAT_COL, "cluster")
_p3_pdf = _p3_pred.toPandas()
_sum = _cluster_summary(_p3_pdf, "cluster", FEATURE_COLS)
_pca_cum, _pca_n = _driver_pca_variance(scaled_sdf)
_panels = _std_panels(
    _p3_pdf, _merge_features(_p3_pdf), _sum,
    _p3_pdf["tsne_1"], _p3_pdf["tsne_2"], opt,
    f"t-SNE embedding  sil={_p3_sil:.3f}", "t-SNE 1", "t-SNE 2",
    _pca_cum, _pca_n, kr, wcss, opt,
    row0=[
        lambda ax: _ax_var(ax, _pca_cum, _pca_n),
        lambda ax: _ax_elbow(ax, kr, wcss, opt, f"Elbow on t-SNE K={opt}"),
        lambda ax: _ax_cl(ax, _p3_pdf["tsne_1"], _p3_pdf["tsne_2"], _p3_pdf["cluster"], opt,
                          f"t-SNE embedding  sil={_p3_sil:.3f}", "t-SNE 1", "t-SNE 2"),
    ],
    row1=[
        lambda ax: _ax_bar(ax, _p3_pdf["cluster"]),
        lambda ax: _ax_anom(ax, _merge_features(_p3_pdf), "cluster", "Failure Rate per Cluster"),
        lambda ax: _ax_heatmap(ax, _merge_features(_p3_pdf), "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    ],
)
_art = _render_dashboard_3x3(
    _p3_pdf, _sum, f"{TARGET_DEVICE} P3 t-SNE+KMeans",
    f"{TARGET_DEVICE} — P3 t-SNE + K-Means  K={opt}  sil={_p3_sil:.3f}",
    f"{TARGET_DEVICE.lower()}_p3_tsne_kmeans.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P3_TSNE_KMeans") as run:
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "TSNE_KMeans"})
    mlflow.log_params({"n_clusters": opt, "tsne_perplexity": TSNE_PERPLEXITY, "tsne_pca_pre": TSNE_PCA_PRE})
    mlflow.log_metrics({"silhouette_score": round(_p3_sil, 4), "silhouette": round(_p3_sil, 4), "n_devices": len(_p3_pdf)})
    mlflow.log_artifact(_art)
    mlflow.spark.log_model(_km_model, "model")
    PIPELINE_RESULTS["P3_TSNE_KMeans"] = {"pdf": _p3_pdf, "sil": _p3_sil, "run_id": run.info.run_id, "flavor": "spark",
        "x": _p3_pdf["tsne_1"], "y": _p3_pdf["tsne_2"]}
print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")



In [ ]:
# ── CELL 12 : [GATE] P4 — PCA + Isolation Forest + Spark K-Means ─────────────
# Matches original pandas Pipeline 5: Spark PCA (95%) -> sklearn IF -> KMeans on inliers
ensure_spark_alive()
_t0 = time.time()
print(f"\n{'='*70}\n  {TARGET_DEVICE} P4: PCA -> Isolation Forest -> K-Means\n{'='*70}")

_pca_sdf, _npc, _cum, _pca_model = pca95_fit_transform(scaled_sdf)
Xp, _pca_meta = collect_pca_matrix(_pca_sdf)
print(f"  Devices: {len(Xp):,}  |  PCA dims (95%): {_npc}")

_if_lbl, _if_score, _if_model = run_isolation_forest(Xp)
_n_anom = int((_if_lbl == -1).sum())
print(f"  IF anomalies removed: {_n_anom:,} ({_n_anom/len(Xp):.1%})")

_inlier_keys = [_pca_meta[DEVICE_KEY][i] for i in range(len(_if_lbl)) if _if_lbl[i] == 1]
_pca_clean = _pca_sdf.filter(F.col(DEVICE_KEY).isin(_inlier_keys))
IF_CACHE.update({"Xp": Xp, "meta": _pca_meta, "if_lbl": _if_lbl, "if_score": _if_score,
                 "npc": _npc, "cum": _cum, "inlier_keys": _inlier_keys})

opt, kr, wcss = spark_elbow(_pca_clean, "pca_features", "PCA+IF+KM")
km = KMeans(k=opt, seed=RANDOM_STATE, featuresCol="pca_features", maxIter=50)
_km_model = km.fit(_pca_clean)
_p4_pred = _km_model.transform(_pca_clean).withColumnRenamed("prediction", "cluster")
_p4_sil = spark_silhouette(_p4_pred, "pca_features", "cluster")
_p4_pdf = sdf_to_pdf(_p4_pred.select(DEVICE_KEY, "cluster", *([TARGET_COL, DEVICE_ID_COL] if TARGET_COL in _p4_pred.columns else [DEVICE_ID_COL])))
_p4_pdf["if_score"] = [_if_score[i] for i in range(len(_if_lbl)) if _if_lbl[i] == 1]
_sum = _cluster_summary(_p4_pdf, "cluster", FEATURE_COLS)
_arr = _p4_pred.select(vector_to_array("pca_features").alias("a"), "cluster").toPandas()
_sx, _sy = _arr["a"].apply(lambda x: x[0]), _arr["a"].apply(lambda x: x[1])
_p4_full = _merge_features(_p4_pdf)
_panels = [
    lambda ax: _ax_var(ax, _cum, _npc),
    lambda ax: _ax_if_hist(ax, _if_score, _if_lbl),
    lambda ax: _ax_cl(ax, _sx, _sy, _arr["cluster"], opt, f"Inlier PC1-PC2  sil={_p4_sil:.3f}", "PC1", "PC2"),
    lambda ax: _ax_elbow(ax, kr, wcss, opt, f"Elbow (inliers) K={opt}"),
    lambda ax: _ax_anom(ax, _p4_full, "cluster", "Failure Rate per Cluster"),
    lambda ax: _ax_heatmap(ax, _p4_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    lambda ax: _ax_bar(ax, _p4_pdf["cluster"]),
    lambda ax: _ax_devcount(ax, _p4_full, "cluster"),
    lambda ax: _ax_summary_text(ax, _sum),
]
_art = _render_dashboard_3x3(
    _p4_pdf, _sum, f"{TARGET_DEVICE} P4 PCA+IF+KMeans",
    f"{TARGET_DEVICE} — P4 PCA+IF+KMeans  n_pca={_npc}  K={opt}  sil={_p4_sil:.3f}",
    f"{TARGET_DEVICE.lower()}_p4_pca_if_kmeans.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P4_PCA_IF_KMeans") as run:
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "PCA_IF_KMeans"})
    mlflow.log_params({"n_clusters": opt, "pca_components": _npc, "if_contamination": IF_CONTAMINATION})
    mlflow.log_metrics({"silhouette_score": round(_p4_sil, 4), "silhouette": round(_p4_sil, 4),
                        "n_anomalies": _n_anom, "n_inliers": len(_p4_pdf)})
    mlflow.log_artifact(_art)
    _p4_pipe = spark_full_pipeline(_pca_model, _km_model)
    mlflow.spark.log_model(_p4_pipe, "model")
    mlflow.sklearn.log_model(_if_model, "if_model")
    PIPELINE_RESULTS["P4_PCA_IF_KMeans"] = {"pdf": _p4_pdf, "sil": _p4_sil, "run_id": run.info.run_id, "flavor": "spark"}
print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")



In [ ]:
# ── CELL 13 : [GATE] P5 — PCA + IF → t-SNE → Spark K-Means ───────────────────
# Isolation Forest on PCA space, then t-SNE on inliers, then KMeans on t-SNE coords
ensure_spark_alive()
_t0 = time.time()
print(f"\n{'='*70}\n  {TARGET_DEVICE} P5: PCA -> IF -> t-SNE -> K-Means\n{'='*70}")

if IF_CACHE.get("Xp") is not None:
    print("  [IF] Reusing Isolation Forest results from P4")
    Xp = IF_CACHE["Xp"]
    _pca_meta = IF_CACHE["meta"]
    _if_lbl = IF_CACHE["if_lbl"]
    _if_score = IF_CACHE["if_score"]
    _npc = IF_CACHE["npc"]
    _cum = IF_CACHE["cum"]
else:
    print("  [IF] P4 not run — computing PCA + Isolation Forest now")
    _pca_sdf, _npc, _cum, _pca_model = pca95_fit_transform(scaled_sdf)
    Xp, _pca_meta = collect_pca_matrix(_pca_sdf)
    _if_lbl, _if_score, _ = run_isolation_forest(Xp)

_inlier_idx = [i for i in range(len(_if_lbl)) if _if_lbl[i] == 1]
Xp_in = Xp[_inlier_idx]
_meta_in = {k: [_pca_meta[k][i] for i in _inlier_idx] for k in _pca_meta}
_n_anom = len(_if_lbl) - len(_inlier_idx)
print(f"  PCA dims: {_npc}  |  IF inliers: {len(_inlier_idx):,}  |  outliers: {_n_anom:,}")

_p5_tsne_pdf, _ = get_or_compute_tsne(
    Xp_in, _meta_in, cache_key="pca_if_inliers", prefix="tsne", pre_pca=False)

tsne_sdf = pdf_tsne_to_spark(_p5_tsne_pdf)
opt, kr, wcss = spark_elbow(tsne_sdf, TSNE_FEAT_COL, "IF+t-SNE+KM")
km = KMeans(k=opt, seed=RANDOM_STATE, featuresCol=TSNE_FEAT_COL, maxIter=50)
_km_model = km.fit(tsne_sdf)
_p5_pred = _km_model.transform(tsne_sdf).withColumnRenamed("prediction", "cluster")
_p5_sil = spark_silhouette(_p5_pred, TSNE_FEAT_COL, "cluster")
_p5_pdf = _p5_pred.toPandas()
_p5_pdf["if_score"] = [_if_score[i] for i in _inlier_idx]
_sum = _cluster_summary(_p5_pdf, "cluster", FEATURE_COLS)
_p5_full = _merge_features(_p5_pdf)
_panels = [
    lambda ax: _ax_var(ax, _cum, _npc),
    lambda ax: _ax_if_hist(ax, _if_score, _if_lbl, title="IF scores (before t-SNE)"),
    lambda ax: _ax_elbow(ax, kr, wcss, opt, f"Elbow on t-SNE K={opt}"),
    lambda ax: _ax_cl(ax, _p5_pdf["tsne_1"], _p5_pdf["tsne_2"], _p5_pdf["cluster"], opt,
                      f"IF inliers in t-SNE  sil={_p5_sil:.3f}", "t-SNE 1", "t-SNE 2"),
    lambda ax: _ax_bar(ax, _p5_pdf["cluster"]),
    lambda ax: _ax_anom(ax, _p5_full, "cluster", "Failure Rate per Cluster"),
    lambda ax: _ax_heatmap(ax, _p5_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    lambda ax: _ax_devcount(ax, _p5_full, "cluster"),
    lambda ax: _ax_summary_text(ax, _sum),
]
_art = _render_dashboard_3x3(
    _p5_pdf, _sum, f"{TARGET_DEVICE} P5 IF+t-SNE+KMeans",
    f"{TARGET_DEVICE} — P5 IF→t-SNE→KMeans  n_pca={_npc}  K={opt}  sil={_p5_sil:.3f}",
    f"{TARGET_DEVICE.lower()}_p5_if_tsne_kmeans.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P5_IF_TSNE_KMeans") as run:
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "IF_TSNE_KMeans"})
    mlflow.log_params({"n_clusters": opt, "pca_components": _npc, "if_contamination": IF_CONTAMINATION})
    mlflow.log_metrics({"silhouette_score": round(_p5_sil, 4), "silhouette": round(_p5_sil, 4),
                        "n_inliers": len(_p5_pdf), "n_anomalies": _n_anom})
    mlflow.log_artifact(_art)
    mlflow.spark.log_model(_km_model, "model")
    PIPELINE_RESULTS["P5_IF_TSNE_KMeans"] = {"pdf": _p5_pdf, "sil": _p5_sil, "run_id": run.info.run_id, "flavor": "spark",
        "x": _p5_pdf["tsne_1"], "y": _p5_pdf["tsne_2"]}
print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")



In [ ]:
# ── CELL 14 : [GATE] P6 — DBSCAN (Spark vectors -> driver) ────────────────────
# DBSCAN has no Spark ML equivalent; vectors prepared in Spark, clustering on driver
ensure_spark_alive()
_t0 = time.time()
print(f"\n{'='*70}\n  {TARGET_DEVICE} P6: DBSCAN (Spark-prepared vectors)\n{'='*70}")

X, meta = collect_feature_matrix(scaled_sdf)
pdf_db = pd.DataFrame({DEVICE_KEY: meta[DEVICE_KEY]})
if TARGET_COL in meta:
    pdf_db[TARGET_COL] = meta[TARGET_COL]
if DEVICE_ID_COL in meta:
    pdf_db[DEVICE_ID_COL] = meta[DEVICE_ID_COL]

_nbrs = NearestNeighbors(n_neighbors=DBSCAN_K_NEIGHBORS).fit(X)
_kdist = np.sort(_nbrs.kneighbors(X)[0][:, -1])[::-1]
_kl = KneeLocator(range(len(_kdist)), _kdist, curve="convex", direction="decreasing")
_eps = float(_kdist[_kl.knee]) if _kl.knee else float(np.percentile(_kdist, 10))
print(f"  Auto eps (k-dist knee): {_eps:.4f}")

pdf_db["cluster"] = DBSCAN(eps=_eps, min_samples=DBSCAN_MIN_SAMPLES, n_jobs=-1).fit_predict(X)
_n_cl = len(set(pdf_db["cluster"].unique()) - {-1})
_n_noise = (pdf_db["cluster"] == -1).sum()
print(f"  Clusters: {_n_cl}  |  Noise: {_n_noise:,} ({_n_noise/len(pdf_db):.1%})")
_mask = pdf_db["cluster"] != -1
_sil = silhouette_score(X[_mask], pdf_db.loc[_mask, "cluster"]) if _n_cl > 1 else float("nan")
print(f"  Silhouette (excl noise): {_sil:.4f}" if _sil == _sil else "  Silhouette: N/A")

_sum = _cluster_summary(pdf_db, "cluster", FEATURE_COLS)
from sklearn.decomposition import PCA as skPCA
X2 = skPCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
_pca_cum, _pca_n = _driver_pca_variance(scaled_sdf)
_db_full = _merge_features(pdf_db)
_panels = [
    lambda ax: _ax_kdist(ax, _kdist, _kl.knee, _eps),
    lambda ax: _ax_var(ax, _pca_cum, _pca_n),
    lambda ax: _ax_cl(ax, X2[:, 0], X2[:, 1], pdf_db["cluster"], _n_cl + 1, "PCA-2D DBSCAN", "PC1", "PC2"),
    lambda ax: _ax_bar(ax, pdf_db["cluster"]),
    lambda ax: _ax_anom(ax, _db_full, "cluster", "Failure Rate per Cluster"),
    lambda ax: _ax_heatmap(ax, _db_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    lambda ax: _ax_devcount(ax, _db_full, "cluster"),
    lambda ax: _ax_timeline(ax, _db_full, "cluster"),
    lambda ax: _ax_summary_text(ax, _sum),
]
_art = _render_dashboard_3x3(
    pdf_db, _sum, f"{TARGET_DEVICE} P6 DBSCAN",
    f"{TARGET_DEVICE} — P6 DBSCAN  eps={_eps:.3f}  clusters={_n_cl}  noise={_n_noise:,}",
    f"{TARGET_DEVICE.lower()}_p6_dbscan.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P6_DBSCAN"):
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "DBSCAN"})
    mlflow.log_params({"eps": round(_eps, 4), "min_samples": DBSCAN_MIN_SAMPLES})
    mlflow.log_metrics({"silhouette": round(float(_sil) if _sil==_sil else 0, 4), "n_clusters": _n_cl, "n_noise": int(_n_noise)})
    mlflow.log_artifact(_art)
PIPELINE_RESULTS["P6_DBSCAN"] = {"pdf": pdf_db, "sil": _sil, "x": X2[:, 0], "y": X2[:, 1]}
print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")



In [ ]:
# ── CELL 15 : [GATE] P7 — Spark Gaussian Mixture Model ────────────────────────
ensure_spark_alive()
_t0 = time.time()
print(f"\n{'='*70}\n  {TARGET_DEVICE} P7: Spark GMM (BIC search)\n{'='*70}")

_bics, _kr = [], list(range(K_MIN, K_MAX + 1))
print("  BIC search:", end=" ", flush=True)
for k in _kr:
    gmm = GaussianMixture(k=k, seed=RANDOM_STATE, featuresCol=FEAT_COL, maxIter=100)
    m = gmm.fit(scaled_sdf)
    ll = m.summary.logLikelihood
    n, d = scaled_sdf.count(), len(FEATURE_COLS)
    # BIC = -2*LL + p*ln(n), p approx k*(d + d*(d+1)/2) for full covariance
    p = k * (2 * d + d * (d + 1) / 2)
    bic = -2 * ll + p * np.log(n)
    _bics.append(bic)
    print(f"K={k}({bic:.0f})", end=" ", flush=True)
print()
_opt = int(_kr[int(np.argmin(_bics))])
print(f"  Optimal K (min BIC): {_opt}")

gmm = GaussianMixture(k=_opt, seed=RANDOM_STATE, featuresCol=FEAT_COL, maxIter=200)
_gmm_model = gmm.fit(scaled_sdf)
_p6_pred = _gmm_model.transform(scaled_sdf).withColumnRenamed("prediction", "cluster")
_p6_sil = spark_silhouette(_p6_pred, FEAT_COL, "cluster")
_p6_pdf = sdf_to_pdf(_p6_pred.select(DEVICE_KEY, "cluster", "probability", *([TARGET_COL, DEVICE_ID_COL] if TARGET_COL in _p6_pred.columns else [DEVICE_ID_COL])))
# GMM confidence = max component probability
_probs = _p6_pred.select(vector_to_array("probability").alias("p")).toPandas()
_p6_pdf["gmm_confidence"] = _probs["p"].apply(lambda x: float(max(x)))

_sum = _cluster_summary(_p6_pdf, "cluster", FEATURE_COLS)
_pca_cum, _pca_n = _driver_pca_variance(scaled_sdf)
_pca2 = PCA(k=2, inputCol=FEAT_COL, outputCol="pca2").fit(scaled_sdf)
_arr = _pca2.transform(_p6_pred.select(DEVICE_KEY, FEAT_COL)).join(_p6_pred.select(DEVICE_KEY, "cluster"), DEVICE_KEY)
_arr = _arr.select(vector_to_array("pca2").alias("a"), "cluster").toPandas()
_sx, _sy = _arr["a"].apply(lambda x: x[0]), _arr["a"].apply(lambda x: x[1])
_gmm_full = _merge_features(_p6_pdf)
_panels = [
    lambda ax: _ax_bic(ax, _kr, _bics, _opt),
    lambda ax: _ax_var(ax, _pca_cum, _pca_n),
    lambda ax: _ax_cl(ax, _sx, _sy, _arr["cluster"], _opt, f"PCA-2D GMM  sil={_p6_sil:.3f}", "PC1", "PC2"),
    lambda ax: _ax_bar(ax, _p6_pdf["cluster"]),
    lambda ax: _ax_anom(ax, _gmm_full, "cluster", "Failure Rate per Cluster"),
    lambda ax: _ax_heatmap(ax, _gmm_full, "cluster", FEATURE_COLS, "Feature Means Heatmap (top 12)"),
    lambda ax: (ax.hist(_p6_pdf["gmm_confidence"], bins=30, color="#1565C0", alpha=0.8),
                ax.set(title="GMM Confidence", xlabel="Max Prob")),
    lambda ax: _ax_devcount(ax, _gmm_full, "cluster"),
    lambda ax: _ax_summary_text(ax, _sum),
]
_art = _render_dashboard_3x3(
    _p6_pdf, _sum, f"{TARGET_DEVICE} P7 GMM",
    f"{TARGET_DEVICE} — P7 GMM  K={_opt}  BIC={min(_bics):.0f}  sil={_p6_sil:.3f}",
    f"{TARGET_DEVICE.lower()}_p7_gmm.png", _panels)[0]
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_P7_GMM") as run:
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "GMM"})
    mlflow.log_params({"n_clusters": _opt})
    mlflow.log_metrics({"silhouette_score": round(_p6_sil, 4), "silhouette": round(_p6_sil, 4), "bic": round(min(_bics), 2),
                        "mean_confidence": round(float(_p6_pdf["gmm_confidence"].mean()), 4)})
    mlflow.log_artifact(_art)
    _gmm_pipe = spark_full_pipeline(_gmm_model)
    mlflow.spark.log_model(_gmm_pipe, "model")
    PIPELINE_RESULTS["P7_GMM"] = {"pdf": _p6_pdf, "sil": _p6_sil, "run_id": run.info.run_id, "flavor": "spark"}
print(f"  Elapsed: {(time.time()-_t0)/60:.2f} min")



In [ ]:
# ── CELL 16 : t-SNE comparison dashboard (all pipelines) ───────────────────────
# Shared t-SNE embedding; reuses cache from P3 when available
ensure_spark_alive()
print(f"\n{'='*70}\n  t-SNE Comparison Dashboard — {TARGET_DEVICE}\n{'='*70}")

X, meta = collect_feature_matrix(scaled_sdf)
pdf_base, _ = get_or_compute_tsne(X, meta, cache_key="scaled_features", prefix="tsne")

_merge_map = {
    "P1_KMeans": _p1_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P1"}),
    "P2_PCA_KM": _p2_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P2"}),
    "P3_tSNE_KM": _p3_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P3"}),
    "P4_PCA_IF_KM": _p4_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P4"}),
    "P5_IF_tSNE_KM": _p5_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P5"}),
    "P6_DBSCAN": pdf_db[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P6"}),
    "P7_GMM": _p6_pdf[[DEVICE_KEY, "cluster"]].rename(columns={"cluster": "cl_P7"}),
}
for name, mdf in _merge_map.items():
    pdf_base = pdf_base.merge(mdf, on=DEVICE_KEY, how="left")

if TARGET_COL in meta:
    pdf_base[TARGET_COL] = meta[TARGET_COL]

_panels = [
    ("P1 Spark KMeans", "cl_P1"),
    ("P2 PCA+KMeans", "cl_P2"),
    ("P3 t-SNE+KMeans", "cl_P3"),
    ("P4 PCA+IF+KMeans", "cl_P4"),
    ("P5 IF→t-SNE+KMeans", "cl_P5"),
    ("P6 DBSCAN", "cl_P6"),
    ("P7 GMM", "cl_P7"),
]
if TARGET_COL in pdf_base.columns:
    _panels.append(("Failure Rate (target)", TARGET_COL))

fig, axes = plt.subplots(3, 3, figsize=(22, 18))
fig.suptitle(f"{TARGET_DEVICE} — t-SNE Comparison Dashboard (shared embedding)", fontsize=14, fontweight="bold")
for ax, (title, col) in zip(axes.flatten(), _panels):
    if col == TARGET_COL:
        sc = ax.scatter(pdf_base["tsne_1"], pdf_base["tsne_2"], c=pdf_base[col], cmap="RdYlGn_r", alpha=0.5, s=8)
        plt.colorbar(sc, ax=ax, shrink=0.7, label="fail rate")
        ax.set_title(title)
    else:
        cl = pdf_base[col].fillna(-99).astype(int)
        nc = len(cl.unique())
        _ax_cl(ax, pdf_base["tsne_1"], pdf_base["tsne_2"], cl, nc, title, "t-SNE 1", "t-SNE 2")
for ax in axes.flatten()[len(_panels):]:
    ax.axis("off")
fig.tight_layout()
_art = _save(fig, f"{TARGET_DEVICE.lower()}_tsne_comparison_dashboard.png")
with mlflow.start_run(run_name=f"{TARGET_DEVICE}_TSNE_Comparison"):
    mlflow.set_tags({**COMMON_TAGS, "pipeline": "TSNE_comparison"})
    mlflow.log_artifact(_art)
print("t-SNE comparison dashboard saved.")



In [ ]:
# ── CELL 17 : MLflow Model Registry — GATE champion (best silhouette) ────────
import boto3
from botocore.exceptions import ClientError
from mlflow.tracking import MlflowClient

_mlflow_client = MlflowClient()
_sm_client = boto3.client("sagemaker", region_name=REGION)
_today = pd.Timestamp.today().strftime("%Y-%m-%d")

_candidates = [(k, v) for k, v in PIPELINE_RESULTS.items()
               if v.get("run_id") and v.get("sil") is not None and v["sil"] == v["sil"]]
if not _candidates:
    _exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if _exp:
        _runs = _mlflow_client.search_runs(
            experiment_ids=[_exp.experiment_id],
            filter_string=f"tags.device_type = 'GATE' and attributes.start_time >= '{_today}'",
            order_by=["metrics.silhouette_score DESC"],
            max_results=10,
        )
        if _runs:
            _best = _runs[0]
            _champ_name = _best.data.tags.get("pipeline", "unknown")
            _champ_run_id = _best.info.run_id
            _champ_sil = float(_best.data.metrics.get("silhouette_score", 0))
            _champ_flavor = _best.data.tags.get("engine", "spark")
        else:
            _champ_name = _champ_run_id = _champ_sil = _champ_flavor = None
    else:
        _champ_name = _champ_run_id = _champ_sil = _champ_flavor = None
else:
    _champ_name, _champ = max(_candidates, key=lambda x: x[1]["sil"])
    _champ_run_id = _champ["run_id"]
    _champ_sil = _champ["sil"]
    _champ_flavor = _champ.get("flavor", "spark")

print(f"\n{'='*70}\n  GATE Model Registry (PySpark)\n{'='*70}")
if not _champ_run_id:
    print("  [WARN] No champion run with logged model — run P1–P5/P7 first")
else:
    print(f"  Champion pipeline : {_champ_name}")
    print(f"  Silhouette        : {_champ_sil:.4f}")
    print(f"  Run ID            : {_champ_run_id}")
    print(f"  Flavor            : {_champ_flavor}")
    _model_uri = f"runs:/{_champ_run_id}/model"
    try:
        _mv = mlflow.register_model(model_uri=_model_uri, name=MLFLOW_MODEL_NAME)
        print(f"  MLflow registered : {MLFLOW_MODEL_NAME} v{_mv.version}")
        for _tk, _tv in {
            "device_type": "GATE", "pipeline": str(_champ_name), "flavor": str(_champ_flavor),
            "silhouette_score": f"{_champ_sil:.4f}", "run_date": _today, "engine": "pyspark",
        }.items():
            _mlflow_client.set_model_version_tag(MLFLOW_MODEL_NAME, _mv.version, _tk, _tv)
        for _ev in _mlflow_client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'"):
            if _ev.current_stage == "Staging" and _ev.version != _mv.version:
                _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _ev.version, "Archived")
        _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _mv.version, "Staging")
        print(f"  Promoted          : {MLFLOW_MODEL_NAME} v{_mv.version} -> Staging")
    except Exception as _e:
        print(f"  [WARN] MLflow register_model failed: {_e}")
        _mv = None

    try:
        _sm_client.describe_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME)
    except ClientError:
        _sm_client.create_model_package_group(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageGroupDescription=f"PS4 GATE fault clustering (PySpark) — {MLFLOW_MODEL_NAME}",
        )
        print(f"  Created SM group  : {MODEL_REGISTRY_NAME}")

    if _mv is not None:
        try:
            _pkg = _sm_client.create_model_package(
                ModelPackageGroupName=MODEL_REGISTRY_NAME,
                ModelPackageDescription=f"GATE champion {_champ_name} sil={_champ_sil:.4f} (PySpark)",
                ModelApprovalStatus="PendingManualApproval",
                MetadataProperties={"GeneratedBy": "PS4_FaultClustering_GATE_PySpark"},
                CustomerMetadataProperties={
                    "device_type": "GATE", "pipeline": str(_champ_name),
                    "silhouette_score": f"{_champ_sil:.4f}", "mlflow_version": str(_mv.version),
                    "engine": "pyspark",
                },
                ModelMetrics=[{
                    "Name": "silhouette_score", "Value": float(_champ_sil),
                    "Type": "number", "Unit": "none",
                }],
            )
            print(f"  SM Model Package  : {_pkg['ModelPackageArn']}")
        except Exception as _e:
            print(f"  [WARN] SageMaker Model Package failed: {_e}")



In [ ]:
# ── CELL 18 : Run summary ──────────────────────────────────────────────────────
print("=" * 70)
print(f"PS4 — {TARGET_DEVICE} Fault Clustering (PySpark) — Complete")
print("=" * 70)
print(f"  Devices clustered : {scaled_sdf.count():,}")
print(f"  Features          : {len(FEATURE_COLS)}")
print(f"  Output dir        : {OUT_DIR}")
print()
print("Pipeline silhouette scores:")
for name, res in PIPELINE_RESULTS.items():
    sil = res.get("sil")
    print(f"  {name:25s}: {sil:.4f}" if sil == sil else f"  {name:25s}: N/A")
print()
print("Artifacts:")
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".png"):
        print(f"  {f}")
print()
print(f"MLflow experiment: {MLFLOW_EXPERIMENT}")
print(f"Model registry   : {MLFLOW_MODEL_NAME}")
_reg = [(k, v) for k, v in PIPELINE_RESULTS.items() if v.get("run_id") and v.get("sil") == v.get("sil")]
if _reg:
    _cn, _cv = max(_reg, key=lambda x: x[1]["sil"])
    print(f"Champion (local)  : {_cn}  sil={_cv['sil']:.4f}  run={_cv['run_id'][:8]}...")

